### 1. Connect to PostgreSQL and prepare data for generation.
    Use `psycopg2` to connect to PostgreSQL, then convert the data into a DataFrame.

In [2]:
import pandas as pd
import psycopg2

In [3]:
conn = psycopg2.connect(
    dbname = "gym_exercise",
    user = "postgres",
    password="password",
    host="localhost"
)

users = "SELECT * FROM users"
users_feedback = "SELECT * FROM users_feedback"
workout_programs = "SELECT * FROM workout_programs"

df_users = pd.read_sql(users, conn)
df_users_feedback = pd.read_sql(users_feedback, conn)
df_workout_programs = pd.read_sql(workout_programs, conn)

C:\Users\yare\AppData\Local\Temp\ipykernel_5276\3397220748.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_users = pd.read_sql(users, conn)
C:\Users\yare\AppData\Local\Temp\ipykernel_5276\3397220748.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_users_feedback = pd.read_sql(users_feedback, conn)
C:\Users\yare\AppData\Local\Temp\ipykernel_5276\3397220748.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_workout_programs = pd.read_sql(workout_programs, conn)


### 2. Combine `df_users`, `df_users_feedback`, `df_workout_programs` into one table.

In [4]:
user_workout = pd.merge(df_workout_programs, df_users, on='program_id', how='left')

In [5]:
print(user_workout.columns)

Index(['program_id', 'program_name', 'day', 'exercise_id', 'name',
       'muscle_group', 'sets', 'min_reps', 'max_reps', 'difficulty', 'user_id',
       'age', 'gender', 'experience_level', 'experience_numeric',
       'days_per_week', 'goal'],
      dtype='object')


In [6]:
full_data = pd.merge(user_workout, df_users_feedback, on=['user_id', 'program_id'], how = 'left')

In [7]:
print(full_data.columns)

Index(['program_id', 'program_name', 'day', 'exercise_id', 'name',
       'muscle_group', 'sets', 'min_reps', 'max_reps', 'difficulty', 'user_id',
       'age', 'gender', 'experience_level', 'experience_numeric',
       'days_per_week', 'goal', 'start_weight_kg', 'end_weight_kg',
       'goal_achieved', 'user_rating', 'comments'],
      dtype='object')


In [8]:
full_data

,program_id,program_name,day,exercise_id,name,muscle_group,sets,min_reps,max_reps,difficulty,...,gender,experience_level,experience_numeric,days_per_week,goal,start_weight_kg,end_weight_kg,goal_achieved,user_rating,comments
0,1,Full Body A,Day 1,28,Bench Press,Chest_Base,3,8,10,beginner,...,male,beginner,1,3,mass,75.0,78.0,yes,5.0,"Great program, gained mass"
1,1,Full Body A,Day 1,28,Bench Press,Chest_Base,3,8,10,beginner,...,female,beginner,1,3,weight_loss,68.0,65.0,yes,4.0,Good for beginners
2,1,Full Body A,Day 1,28,Bench Press,Chest_Base,3,8,10,beginner,...,male,beginner,1,3,mass,71.0,74.0,yes,4.0,Solid program
3,1,Full Body A,Day 1,28,Bench Press,Chest_Base,3,8,10,beginner,...,female,beginner,1,3,toning,69.0,66.0,yes,4.0,Good beginner program
4,1,Full Body A,Day 1,28,Bench Press,Chest_Base,3,8,10,beginner,...,male,beginner,1,3,strength,70.0,72.0,no,4.0,Good foundation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,9,Body Part B,Day 5 Chest/Back,21,Bent-Over Barbell Row,Back_Latissimus Dorsi,3,8,10,intermediate,...,female,advanced,3,5,toning,55.0,56.0,no,4.0,Enjoyed it
2553,9,Body Part B,Day 5 Chest/Back,21,Bent-Over Barbell Row,Back_Latissimus Dorsi,3,8,10,intermediate,...,female,advanced,3,5,toning,55.0,56.0,no,4.0,Enjoyed
2554,9,Body Part B,Day 5 Chest/Back,26,Seated Cable Row,Rhomboids,3,10,12,beginner,...,female,advanced,3,5,toning,54.0,55.0,no,4.0,Enjoyed the variety
2555,9,Body Part B,Day 5 Chest/Back,26,Seated Cable Row,Rhomboids,3,10,12,beginner,...,female,advanced,3,5,toning,55.0,56.0,no,4.0,Enjoyed it


In [11]:
print(full_data['goal'].unique())

['mass' 'weight_loss' 'toning' 'strength']


### 3. Drop unnecessary columns

In [30]:
full_data = full_data.drop(['program_id', 'exercise_id', 'user_id', 'experience_level','comments', 'name'], axis=1)

In [31]:
full_data

,program_name,day,muscle_group,sets,min_reps,max_reps,difficulty,age,gender,experience_numeric,days_per_week,goal,start_weight_kg,end_weight_kg,goal_achieved,user_rating
0,Full Body A,Day 1,Chest_Base,3,8,10,beginner,23,male,1,3,mass,75.0,78.0,yes,5.0
1,Full Body A,Day 1,Chest_Base,3,8,10,beginner,24,female,1,3,weight_loss,68.0,65.0,yes,4.0
2,Full Body A,Day 1,Chest_Base,3,8,10,beginner,37,male,1,3,mass,71.0,74.0,yes,4.0
3,Full Body A,Day 1,Chest_Base,3,8,10,beginner,46,female,1,3,toning,69.0,66.0,yes,4.0
4,Full Body A,Day 1,Chest_Base,3,8,10,beginner,28,male,1,3,strength,70.0,72.0,no,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,Body Part B,Day 5 Chest/Back,Back_Latissimus Dorsi,3,8,10,intermediate,41,female,3,5,toning,55.0,56.0,no,4.0
2553,Body Part B,Day 5 Chest/Back,Back_Latissimus Dorsi,3,8,10,intermediate,24,female,3,5,toning,55.0,56.0,no,4.0
2554,Body Part B,Day 5 Chest/Back,Rhomboids,3,10,12,beginner,26,female,3,5,toning,54.0,55.0,no,4.0
2555,Body Part B,Day 5 Chest/Back,Rhomboids,3,10,12,beginner,41,female,3,5,toning,55.0,56.0,no,4.0


In [32]:
full_data.isna().sum()

program_name           0
day                    0
muscle_group           0
sets                   0
min_reps               0
max_reps               0
difficulty             0
age                    0
gender                 0
experience_numeric     0
days_per_week          0
goal                   0
start_weight_kg       32
end_weight_kg         32
goal_achieved         32
user_rating           32
dtype: int64

### 4. Fill missing values in the rows

In [35]:
full_data['goal_achieved'] = full_data['goal_achieved'].fillna(full_data['goal_achieved'].mode()[0])

empty_value = ['start_weight_kg', 'end_weight_kg','user_rating']
full_data[empty_value] = full_data[empty_value].fillna(full_data[empty_value].mean())
full_data.isna().sum()

program_name          0
day                   0
muscle_group          0
sets                  0
min_reps              0
max_reps              0
difficulty            0
age                   0
gender                0
experience_numeric    0
days_per_week         0
goal                  0
start_weight_kg       0
end_weight_kg         0
goal_achieved         0
user_rating           0
dtype: int64

### 5. Saving the original prepared data

In [39]:
full_data.to_csv('gym.csv', index=False) 

### 6. Use `LabelEncoder` for encoding dataset

In [40]:
from sklearn.preprocessing import LabelEncoder

cat_cols = full_data.select_dtypes(include='object').columns.tolist()
print(cat_cols)

encoders = {}
for col in cat_cols:
    if col in full_data.columns:
        le = LabelEncoder()
        full_data[col] = le.fit_transform(full_data[col].astype(str))
        encoders[col] = le

print(full_data.dtypes)

[]
program_name            int64
day                     int64
muscle_group            int64
sets                    int64
min_reps                int64
max_reps                int64
difficulty              int64
age                     int64
gender                  int64
experience_numeric      int64
days_per_week           int64
goal                    int64
start_weight_kg       float64
end_weight_kg         float64
goal_achieved           int64
user_rating           float64
dtype: object


### 7. Dataset after encoding

In [41]:
full_data

,program_name,day,muscle_group,sets,min_reps,max_reps,difficulty,age,gender,experience_numeric,days_per_week,goal,start_weight_kg,end_weight_kg,goal_achieved,user_rating
0,2,0,4,3,8,10,1,23,1,1,3,0,75.0,78.0,1,5.0
1,2,0,4,3,8,10,1,24,0,1,3,3,68.0,65.0,1,4.0
2,2,0,4,3,8,10,1,37,1,1,3,0,71.0,74.0,1,4.0
3,2,0,4,3,8,10,1,46,0,1,3,2,69.0,66.0,1,4.0
4,2,0,4,3,8,10,1,28,1,1,3,1,70.0,72.0,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,1,15,0,3,8,10,2,41,0,3,5,2,55.0,56.0,0,4.0
2553,1,15,0,3,8,10,2,24,0,3,5,2,55.0,56.0,0,4.0
2554,1,15,14,3,10,12,1,26,0,3,5,2,54.0,55.0,0,4.0
2555,1,15,14,3,10,12,1,41,0,3,5,2,55.0,56.0,0,4.0


### 8. For generate more data use `CTGAN`.
Install `ctgan`, `sdv`

In [42]:
!pip install ctgan
!pip install sdv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 9. Use `SingleTableMetadata` for saved information about the table

In [45]:
from ctgan import CTGAN
from sdv.evaluation.single_table import evaluate_quality
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(full_data)

### 10. Create and train `CTGAN`

In [47]:
ctgan = CTGAN(epochs=100, batch_size=500, log_frequency=False,verbose=True)

ctgan.fit(full_data, cat_cols)

Gen. (+00.18) | Discrim. (-00.51): 100%|█████████████████████████████████████████████| 100/100 [00:16<00:00,  6.02it/s]


### 11. Generate 10,000 new synthetic data samples using CTGAN

In [48]:
synthetic_rows = ctgan.sample(10000)

### 12. Evaluate synthetic data quality

In [49]:
quality_report = evaluate_quality(
    real_data=full_data,
    synthetic_data=synthetic_rows,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |█████████████████████████████████████████████████████| 16/16 [00:00<00:00, 30.22it/s]|
Column Shapes Score: 85.94%

(2/2) Evaluating Column Pair Trends: |█████████████████████████████████████████████| 120/120 [00:00<00:00, 288.82it/s]|
Column Pair Trends Score: 51.27%

Overall Score (Average): 68.61%



### 13. Synthetic data 

In [50]:
synthetic_rows

,program_name,day,muscle_group,sets,min_reps,max_reps,difficulty,age,gender,experience_numeric,days_per_week,goal,start_weight_kg,end_weight_kg,goal_achieved,user_rating
0,8,10,3,3,8,12,1,43,0,3,6,0,61.539813,78.459676,1,4.957491
1,6,-1,10,3,12,15,2,32,0,1,3,0,56.204480,70.562958,1,3.943019
2,7,17,9,3,8,10,0,51,0,1,4,3,64.029129,61.399713,1,3.942159
3,6,17,10,3,8,12,2,42,1,1,3,3,75.529754,60.454978,0,2.985313
4,4,12,11,3,10,15,2,13,1,1,4,1,69.814184,80.347199,1,4.954583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,8,0,3,3,12,10,1,22,1,3,6,3,80.595362,79.585499,0,2.971333
9996,3,10,13,3,10,10,1,38,0,1,4,0,57.604077,72.285873,0,4.992373
9997,4,-1,8,3,14,15,1,44,0,1,6,0,80.684114,56.860814,1,3.975273
9998,7,2,10,3,8,10,2,48,0,3,6,0,85.923107,77.497781,1,3.962607


In [51]:
synthetic_rows['day'] = synthetic_rows['day'].replace([-1,-2], 0)

In [52]:
synthetic_rows.isnull().sum()

program_name          0
day                   0
muscle_group          0
sets                  0
min_reps              0
max_reps              0
difficulty            0
age                   0
gender                0
experience_numeric    0
days_per_week         0
goal                  0
start_weight_kg       0
end_weight_kg         0
goal_achieved         0
user_rating           0
dtype: int64

### 14. Combine original dataset with generated dataset

In [53]:
gym_data = pd.concat([full_data,synthetic_rows], ignore_index=True)

In [54]:
gym_data

,program_name,day,muscle_group,sets,min_reps,max_reps,difficulty,age,gender,experience_numeric,days_per_week,goal,start_weight_kg,end_weight_kg,goal_achieved,user_rating
0,2,0,4,3,8,10,1,23,1,1,3,0,75.000000,78.000000,1,5.000000
1,2,0,4,3,8,10,1,24,0,1,3,3,68.000000,65.000000,1,4.000000
2,2,0,4,3,8,10,1,37,1,1,3,0,71.000000,74.000000,1,4.000000
3,2,0,4,3,8,10,1,46,0,1,3,2,69.000000,66.000000,1,4.000000
4,2,0,4,3,8,10,1,28,1,1,3,1,70.000000,72.000000,0,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12552,8,0,3,3,12,10,1,22,1,3,6,3,80.595362,79.585499,0,2.971333
12553,3,10,13,3,10,10,1,38,0,1,4,0,57.604077,72.285873,0,4.992373
12554,4,0,8,3,14,15,1,44,0,1,6,0,80.684114,56.860814,1,3.975273
12555,7,2,10,3,8,10,2,48,0,3,6,0,85.923107,77.497781,1,3.962607


### 15. Save the new dataset


In [56]:
gym_data.to_csv('ctgan_gym.csv', index=False) 